In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
#this is done in order to import tensorflow then keras and later we will use keras's layer ssystem in our model.

In [4]:

i_s = (224,224)
b_s = 32
#these are image size and batch(number of images in one epoch) size respectively.
dir = r"E:\Downloads\Telegram Desktop\plant_disease\all\Crop Diseases\rice"
# now creating treaning set.
train = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "training",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
    
val = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "validation",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
#this is the validation set
#now time for dividing validation set into val and test sets
valbatches = tf.data.experimental.cardinality(val)
test = val.take(valbatches//2)
val= val.skip(valbatches//2)
class_names = train.class_names
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train).numpy())
print("Val batches:", tf.data.experimental.cardinality(val).numpy())
print("Test batches:", tf.data.experimental.cardinality(test).numpy())
# it is better to shuffle trianing set before augmentation
train = train.shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
#previous line means it shuffle in batch of 1000 and cpu automatically loads the next batch while gpu is training model by using prefetch function
val   = val.prefetch(buffer_size=tf.data.AUTOTUNE)
test  = test.prefetch(buffer_size=tf.data.AUTOTUNE)
#now its time for augmentation
aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import optimizers , models, layers
num = len(class_names)

base = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base.trainable = False
from keras.callbacks import EarlyStopping , ReduceLROnPlateau , ModelCheckpoint
inputs = layers.Input((224, 224, 3))
x = aug(inputs)

x = base(x, training = False)
#this is the phase of augmentation and normalization like max norm

x = layers.GlobalAveragePooling2D()(x) #this is pooling
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num, activation = "softmax")(x)
model = models.Model(inputs, outputs)
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3),
    ModelCheckpoint("models/head_stage.h5", save_best_only=True)
]
history_head = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=callbacks
)


Found 2590 files belonging to 3 classes.
Using 2072 files for training.
Found 2590 files belonging to 3 classes.
Using 518 files for validation.
Classes: ['Rice___Brown_Spot', 'Rice___Leaf_Blast', 'Rice___Neck_Blast']
Train batches: 65
Val batches: 9
Test batches: 8
Epoch 1/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 973ms/step - accuracy: 0.6450 - loss: 0.9446

65/65 ━━━━━━━━━━━━━━━━━━━━ 148s 1s/step - accuracy: 0.7428 - loss: 0.6449 - val_accuracy: 0.7405 - val_loss: 0.4617 - learning_rate: 0.0010
Epoch 2/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 921ms/step - accuracy: 0.8324 - loss: 0.4249

65/65 ━━━━━━━━━━━━━━━━━━━━ 115s 1s/step - accuracy: 0.8359 - loss: 0.4033 - val_accuracy: 0.8550 - val_loss: 0.3429 - learning_rate: 0.0010
Epoch 3/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.8538 - loss: 0.3393 - val_accuracy: 0.8053 - val_loss: 0.3504 - learning_rate: 0.0010
Epoch 4/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 902ms/step - accuracy: 0.8629 - loss: 0.3169

65/65 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.8567 - loss: 0.3332 - val_accuracy: 0.8588 - val_loss: 0.3230 - learning_rate: 0.0010
Epoch 5/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 899ms/step - accuracy: 0.8573 - loss: 0.3220

65/65 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.8518 - loss: 0.3373 - val_accuracy: 0.8702 - val_loss: 0.2918 - learning_rate: 0.0010
Epoch 6/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - accuracy: 0.8798 - loss: 0.2853 - val_accuracy: 0.8664 - val_loss: 0.3090 - learning_rate: 0.0010
Epoch 7/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - accuracy: 0.8673 - loss: 0.2928 - val_accuracy: 0.8702 - val_loss: 0.3030 - learning_rate: 0.0010
Epoch 8/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8649 - loss: 0.3007

65/65 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - accuracy: 0.8745 - loss: 0.2859 - val_accuracy: 0.8855 - val_loss: 0.2795 - learning_rate: 3.0000e-04
Epoch 9/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8820 - loss: 0.2742

65/65 ━━━━━━━━━━━━━━━━━━━━ 123s 1s/step - accuracy: 0.8760 - loss: 0.2862 - val_accuracy: 0.8855 - val_loss: 0.2600 - learning_rate: 3.0000e-04
Epoch 10/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - accuracy: 0.8822 - loss: 0.2886 - val_accuracy: 0.8779 - val_loss: 0.3033 - learning_rate: 3.0000e-04


In [5]:
test_loss, test_accuracy = model.evaluate(test)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

8/8 ━━━━━━━━━━━━━━━━━━━━ 8s 945ms/step - accuracy: 0.8750 - loss: 0.2709
Test Loss: 0.2709
Test Accuracy: 87.50%


In [6]:
model.save("rice.keras")